<a href="https://colab.research.google.com/github/Osondu-ifunanya/Habitat-suitability-modeling-using-MaxEnt-and-ensemble-ML-models/blob/main/Habitat%20suitability%20modeling%20using%20ML%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Habitat Suitability Modeling using MaxEnt (approximation) and Ensemble ML
(Synthetic Data)

Workflow:
1. Generate environmental variables
2. Simulate species presence/background data
3. Train MaxEnt-like model (Logistic Regression)
4. Train ensemble ML models (RF, GB)
5. Combine predictions
6. Generate suitability map
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

np.random.seed(42)

# ---------------------------------------
# 1. Generate Synthetic Environmental Data
# ---------------------------------------

size = 150

temperature = np.random.normal(20, 5, (size, size))
precipitation = np.random.normal(1000, 300, (size, size))
elevation = np.random.normal(500, 200, (size, size))
ndvi = np.random.uniform(0.2, 0.8, (size, size))

# ---------------------------------------
# 2. Simulate Species Presence (MaxEnt style)
# ---------------------------------------

def suitability_func(temp, prec, elev, ndvi):
    return (
        np.exp(-(temp - 22)**2 / 30) *
        np.exp(-(prec - 1100)**2 / 200000) *
        np.exp(-(elev - 600)**2 / 80000) *
        ndvi
    )

prob = suitability_func(temperature, precipitation, elevation, ndvi)
prob = prob / prob.max()

presence = np.random.binomial(1, prob)

# ---------------------------------------
# 3. Prepare Dataset
# ---------------------------------------

X = np.stack([
    temperature.flatten(),
    precipitation.flatten(),
    elevation.flatten(),
    ndvi.flatten()
], axis=1)

y = presence.flatten()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Scale for MaxEnt (logistic regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ---------------------------------------
# 4. Train Models
# ---------------------------------------

# MaxEnt approximation
maxent = LogisticRegression(max_iter=1000)
maxent.fit(X_train_scaled, y_train)

# Ensemble models
rf = RandomForestClassifier(n_estimators=200)
gb = GradientBoostingClassifier()

rf.fit(X_train, y_train)
gb.fit(X_train, y_train)

# ---------------------------------------
# 5. Predictions
# ---------------------------------------

maxent_pred = maxent.predict_proba(X_test_scaled)[:,1]
rf_pred = rf.predict_proba(X_test)[:,1]
gb_pred = gb.predict_proba(X_test)[:,1]

# Ensemble prediction
ensemble_pred = (maxent_pred + rf_pred + gb_pred) / 3

# ---------------------------------------
# 6. Evaluation
# ---------------------------------------

print("Model AUC Scores:")
print("MaxEnt:", roc_auc_score(y_test, maxent_pred))
print("Random Forest:", roc_auc_score(y_test, rf_pred))
print("Gradient Boosting:", roc_auc_score(y_test, gb_pred))
print("Ensemble:", roc_auc_score(y_test, ensemble_pred))

# ---------------------------------------
# 7. Full Map Prediction
# ---------------------------------------

X_scaled_full = scaler.transform(X)

maxent_full = maxent.predict_proba(X_scaled_full)[:,1]
rf_full = rf.predict_proba(X)[:,1]
gb_full = gb.predict_proba(X)[:,1]

ensemble_full = (maxent_full + rf_full + gb_full) / 3
suitability_map = ensemble_full.reshape(size, size)

# ---------------------------------------
# 8. Visualization
# ---------------------------------------

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.imshow(presence, cmap="Greens")
plt.title("Species Presence")

plt.subplot(1,2,2)
plt.imshow(suitability_map, cmap="viridis")
plt.title("Habitat Suitability (Ensemble)")
plt.colorbar()

plt.tight_layout()
plt.show()

# ---------------------------------------
# 9. Export Results
# ---------------------------------------

output = pd.DataFrame({
    "Temperature": X[:,0],
    "Precipitation": X[:,1],
    "Elevation": X[:,2],
    "NDVI": X[:,3],
    "Suitability": ensemble_full
})

output.to_excel("habitat_suitability_results.xlsx", index=False)

print("Results exported to habitat_suitability_results.xlsx")